# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# .env file should contain the following variables (already set at workspace root):
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"   # can be same as OPENAI_API_KEY
# TAVILY_API_KEY="YOUR_KEY"
print("Environment variables are loaded from the workspace root .env file")


Environment variables are loaded from the workspace root .env file


In [4]:
from dotenv import load_dotenv, find_dotenv

# find_dotenv() searches up the directory tree to find the .env file
load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print(f"OPENAI_API_KEY loaded: {'yes' if OPENAI_API_KEY else 'NO - check .env'}")
print(f"TAVILY_API_KEY loaded: {'yes' if TAVILY_API_KEY else 'NO - check .env'}")


OPENAI_API_KEY loaded: yes
TAVILY_API_KEY loaded: yes


### VectorDB Instance

In [5]:
# Instantiate the ChromaDB Persistent Client
# PersistentClient saves data to disk so Notebook 02 can reuse the same collection
chroma_client = chromadb.PersistentClient(path="chromadb")

print(f"ChromaDB client created. Collections: {[c.name for c in chroma_client.list_collections()]}")


ChromaDB client created. Collections: ['udaplay']


### Collection

In [6]:
VOCAREUM_BASE_URL = "https://openai.vocareum.com/v1"

# Embedding function using OpenAI text-embedding-3-small via Vocareum proxy
# api_base routes the request through the Udacity workspace OpenAI endpoint
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small",
    api_base=VOCAREUM_BASE_URL if OPENAI_API_KEY.startswith("voc-") else None
)

print("Embedding function created: OpenAI text-embedding-3-small")


Embedding function created: OpenAI text-embedding-3-small


In [7]:
# Delete if exists (safe re-run) then create fresh collection
try:
    chroma_client.delete_collection(name="udaplay")
    print("Existing 'udaplay' collection deleted.")
except Exception:
    pass  # Collection didn't exist yet

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print(f"Collection 'udaplay' created. Documents currently indexed: {collection.count()}")


Existing 'udaplay' collection deleted.
Collection 'udaplay' created. Documents currently indexed: 0


### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Semantic Search Demo

In [9]:
# Demonstrate semantic search on the collection
query_text = "best racing games for PlayStation"

results = collection.query(
    query_texts=[query_text],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Semantic search query: \"{query_text}\"\n")
print(f"Top {len(results['ids'][0])} results:\n{'-' * 60}")

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1
):
    print(f"\n  Result {i}:")
    print(f"    Name:     {meta.get('Name', 'N/A')}")
    print(f"    Platform: {meta.get('Platform', 'N/A')}")
    print(f"    Year:     {meta.get('YearOfRelease', 'N/A')}")
    print(f"    Genre:    {meta.get('Genre', 'N/A')}")
    print(f"    Distance: {dist:.4f}")
    print(f"    Document: {doc[:120]}{'...' if len(doc) > 120 else ''}")

Semantic search query: "best racing games for PlayStation"

Top 3 results:
------------------------------------------------------------

  Result 1:
    Name:     Gran Turismo
    Platform: PlayStation 1
    Year:     1997
    Genre:    Racing
    Distance: 0.7804
    Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a ...

  Result 2:
    Name:     Gran Turismo 5
    Platform: PlayStation 3
    Year:     2010
    Genre:    Racing
    Distance: 0.7919
    Document: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and trac...

  Result 3:
    Name:     Grand Theft Auto: San Andreas
    Platform: PlayStation 2
    Year:     2004
    Genre:    Action-adventure
    Distance: 1.3365
    Document: [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San An...
